In [ ]:
# [auto] project-root setup
import os, sys
from pathlib import Path

# 自动向上查找项目根目录 (含 .gitignore 的文件夹)
_p = Path.cwd().resolve()
while _p != _p.parent and not (_p / '.gitignore').exists():
    _p = _p.parent
PROJECT_ROOT = _p

# 切换 cwd 到项目根, 使所有相对路径 (Stage1_Exploration/, Refined_Results_v4/ 等) 保持有效
os.chdir(PROJECT_ROOT)
# 让 notebooks 能 `from viz_config import VizConfig`
sys.path.insert(0, str(PROJECT_ROOT))

DATA_DIR = PROJECT_ROOT / 'data'
print(f'[setup] PROJECT_ROOT = {PROJECT_ROOT}')


In [ ]:
import pandas as pd
import numpy as np
import os
from pysr import PySRRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score

# ==========================================
# 0. 环境与参数配置 (Configuration)
# ==========================================
# 缩放系数：处理浓度 1e-7 的量级问题，将其映射到 [0, 10] 左右的区间，方便符号回归搜索
SCALE = 1e7
OUTPUT_DIR = "Stage1_Exploration"
if not os.path.exists(OUTPUT_DIR):
    os.makedirs(OUTPUT_DIR)

# PySR 参数设置 (PySR Hyperparameters)
# 这些参数控制了符号回归搜索的广度和深度
PYSR_PARAMS = {
    "niterations": 150,           # 迭代次数：越大搜索越久，结果通常越好
    "populations": 30,            # 种群数量：并行搜索的独立种群数
    "maxsize": 25,                # 公式最大长度：防止生成过于复杂的过拟合公式
    "binary_operators": ["+", "*", "-", "/"], # 允许的二元运算符
    # 允许的一元运算符：包含物理模型中常见的指数、对数、平方根等
    "unary_operators": ["exp", "log", "sqrt", "square", "inv(x)=1/x"],
    "extra_sympy_mappings": {'inv': lambda x: 1/x}, # 自定义算子的 SymPy 映射
    "model_selection": "best",    # 模型选择策略：选择综合评分最高的
    "temp_equation_file": True,   # 保存临时方程文件，防止中断丢失
    "delete_tempfiles": False,    # 保留临时文件用于调试
    "verbosity": 1                # 开启日志，方便观察搜索进展
}

# ==========================================
# 1. 数据加载与预处理 (Data Loading & Preprocessing)
# ==========================================
print("正在加载数据并按 Case 进行 7:3 划分...")
# 读取清洗后的数据集
df = pd.read_csv('data/train_dataset_ready.csv')

# 浓度预缩放 (Scaling)
# 将物理量纲统一调整到数值计算友好的范围
df['C_in'] = df['C_in'] * SCALE
df['C_out'] = df['C_out'] * SCALE

# 按 Case 名字划分 (Group Split)
# 关键步骤：必须按 Case 划分训练/测试集，而不是按样本点随机划分
# 这是为了防止"数据泄露" (Data Leakage)，因为同一个 Case 内的相邻点高度相关
unique_cases = df['Case'].unique()
train_cases, test_cases = train_test_split(unique_cases, test_size=0.3, random_state=42)

# 根据 Case ID 筛选数据
train_df = df[df['Case'].isin(train_cases)].copy()
test_df = df[df['Case'].isin(test_cases)].copy()

print(f"训练集 Case 数: {len(train_cases)}, 测试集 Case 数: {len(test_cases)}")

# 准备特征矩阵 X 和目标向量 y
feature_names = ['V_in', 'C_in', 'Area', 'Distance']
X_train = train_df[feature_names].values
y_train = train_df['C_out'].values
X_test = test_df[feature_names].values
y_test = test_df['C_out'].values

# 数据降采样 (Subsampling)
# 为了兼顾 PySR 的运行性能和搜索深度，从训练集中随机抽取 5000 个样本点
# (符号回归对数据量不敏感，5000个点通常足以捕捉物理规律，过多的数据会显著拖慢进化速度)
np.random.seed(42)
sub_idx = np.random.choice(len(X_train), 5000, replace=False)
X_train_sub = X_train[sub_idx]
y_train_sub = y_train[sub_idx]

# ==========================================
# 2. 运行 PySR (Symbolic Regression Execution)
# ==========================================
print("\n>>> 开始第一阶段：完全数据驱动探索 (Data-Driven Exploration)...")

# 初始化回归器
model = PySRRegressor(**PYSR_PARAMS)

# 开始拟合
# variable_names 用于让输出的公式直接显示物理变量名，而不是 x0, x1...
model.fit(X_train_sub, y_train_sub, variable_names=feature_names)

# ==========================================
# 3. 结果保存与性能评估 (Evaluation & Saving)
# ==========================================
# 1. 保存所有候选公式到 CSV，供后续分析
equations_path = os.path.join(OUTPUT_DIR, "stage1_all_equations.csv")
model.equations_.to_csv(equations_path)

# 2. 在全量测试集上评估泛化性能
# 注意：这里使用的是从未见过的 Test Cases，能真实反映公式的物理泛化能力
y_pred = model.predict(X_test)
r2 = r2_score(y_test, y_pred)

print("\n" + "="*40)
print(f"第一阶段探索完成！")
print(f"全量测试集 (新Case) R2: {r2:.5f}")
print(f"所有候选公式已保存至: {equations_path}")
print("="*40)

# 输出 PySR 认为的最佳公式 (Pareto Front 上的最优解)
print("\n推荐的最佳公式 (SymPy 格式):")
print(model.sympy())

正在加载数据并按 Case 进行 7:3 划分...
训练集 Case 数: 350, 测试集 Case 数: 150

>>> 开始第一阶段：完全数据驱动探索 (150 Iterations)...


D:\Users\A\anaconda3\envs\py310_llm\lib\site-packages\pysr\sr.py:2811: UserWarning: Note: it looks like you are running in Jupyter. The progress bar will be turned off.
  warnings.warn(
[ Info: Started!



Expressions evaluated per second: 2.260e+04
Progress: 121 / 4500 total iterations (2.689%)
════════════════════════════════════════════════════════════════════════════════════════════════════
───────────────────────────────────────────────────────────────────────────────────────────────────
Complexity  Loss       Score      Equation
1           3.930e+01  0.000e+00  y = 5.5974
2           3.331e+01  1.654e-01  y = sqrt(C_in)
3           9.494e+00  1.255e+00  y = C_in * 0.50729
5           9.346e+00  7.840e-03  y = (C_in + 1.0821) * 0.48418
12          5.162e+00  8.480e-02  y = (C_in * sqrt(2.2656 / ((Distance * 0.012199) + 1.8696)...
                                      )) + 1.2325
13          2.667e+00  6.605e-01  y = ((1.0195 / sqrt(sqrt(1.541 + (Distance * 0.044316)))) ...
                                      * C_in) + 0.25298
14          1.234e+00  7.707e-01  y = ((1.819 / sqrt(((Distance * 0.028222) - 0.92633) + 3.8...
                                      482)) * C_in) + 0.529

[ Info: Final population:
[ Info: Results saved to:


───────────────────────────────────────────────────────────────────────────────────────────────────
Complexity  Loss       Score      Equation
1           3.930e+01  0.000e+00  y = 5.5974
2           3.331e+01  1.654e-01  y = sqrt(C_in)
3           9.494e+00  1.255e+00  y = C_in / 1.9713
5           9.346e+00  7.840e-03  y = (C_in + 1.0821) * 0.48418
6           8.915e+00  4.725e-02  y = (log(Area) * 0.14267) * C_in
7           1.079e+00  2.112e+00  y = (C_in * 445.07) / (Distance + 456.55)
9           5.289e-01  3.565e-01  y = (C_in * 12.767) / ((Distance / Area) + 13.217)
11          4.212e-01  1.138e-01  y = (C_in * (V_in + 12.195)) / ((Distance / Area) + 13.14)
13          2.669e-01  2.281e-01  y = (C_in * 9.8782) / ((Distance / ((V_in * 18.961) + Area...
                                      )) + 10.138)
14          2.667e-01  9.553e-04  y = C_in * (9.6892 / ((Distance * inv(Area + (V_in / 0.049...
                                      727))) + 9.9212))
15          2.666e-01  1.76